# WMT19 TTS LongCat Dataset Check

这个 notebook 用来在开发机上直接查看当前 speech-to-speech 使用的 WMT19 zh-en TTS/LongCat store。

默认入口来自 `zhuyin.datasets.wmt19_tts.wmt19_tts_longcat()`。notebook 不修改 `sys.path`，运行前应在 kernel 环境里安装 `workspace` 及依赖包。

可用环境变量覆盖：

- `STATIC_HOME`，缺失时使用复旦共享默认 `/mnt/pami202/zhuyin` 并发 warning。


In [ ]:
from __future__ import annotations

import torch
from IPython.display import Audio, display

from anydataset import AudioView, Modality, Role
from anytrain.codec import LongCatAudioCodec
from zhuyin.datasets.wmt19_tts import wmt19_tts_longcat


## Load Dataset

如果当前机器上的数据集根目录不同，可以在下一格直接传 `dataset_dir=...`，默认从 `$STATIC_HOME/datasets/wmt19_tts/longcat-delta` 解析；`STATIC_HOME` 缺失时使用 `/mnt/pami202/zhuyin`。

In [ ]:
dataset = wmt19_tts_longcat()

print(dataset.spec)
print("len =", len(dataset))


## Inspect One Sample

改 `index` 可以看其它样本。训练契约要求 source/target 两侧都有 `AudioView.LONGCAT`，里面至少包含 `semantic_codes` 和 `acoustic_codes`。

这里不打印具体 code，只看形状。

In [ ]:
index = 0
sample = dataset[index]

print("sample keys:")
for key in sample:
    print("-", key)


In [ ]:
def describe_shape(value):
    shape = getattr(value, "shape", None)
    dtype = getattr(value, "dtype", None)
    if shape is not None:
        return {"type": type(value).__name__, "shape": tuple(shape), "dtype": str(dtype)}
    if isinstance(value, list | tuple):
        return {"type": type(value).__name__, "shape": (len(value),)}
    return {"type": type(value).__name__, "shape": None}


def longcat_view(role: Role):
    audio = sample[(role, Modality.AUDIO)]
    return audio.views[AudioView.LONGCAT]


def describe_longcat(role: Role):
    audio = sample[(role, Modality.AUDIO)]
    print(f"\n[{role.value}] view keys:", list(audio.views))
    print(f"[{role.value}] meta keys:", list(audio.meta))
    for key, value in longcat_view(role).items():
        print(f"[{role.value}] {key}: {describe_shape(value)}")


describe_longcat(Role.SOURCE)
describe_longcat(Role.TARGET)


## Decode Waveform

用 LongCat decoder 从 `semantic_codes` 和 `acoustic_codes` 还原 source/target 波形。默认使用 `16k_4codebooks`，对应 sample rate 为 16000。

In [ ]:
decoder = "16k_4codebooks"
sample_rate = 16000
device = "cuda" if torch.cuda.is_available() else "cpu"

codec = LongCatAudioCodec.from_pretrained(decoders=(decoder,), device=device).eval()


@torch.no_grad()
def decode_role(role: Role):
    view = longcat_view(role)
    semantic = torch.as_tensor(view["semantic_codes"], dtype=torch.long).reshape(1, -1)
    acoustic = torch.as_tensor(view["acoustic_codes"], dtype=torch.long)
    if acoustic.dim() == 2:
        acoustic = acoustic.unsqueeze(0)
    if acoustic.dim() != 3:
        raise ValueError(f"{role.value} acoustic_codes must have shape [nq, time] or [batch, nq, time].")
    if semantic.shape[-1] != acoustic.shape[-1]:
        raise ValueError(
            f"{role.value} semantic/acoustic length mismatch: "
            f"semantic={semantic.shape[-1]} acoustic={acoustic.shape[-1]}"
        )
    waveform = codec.decode(semantic, acoustic, decoder=decoder).detach().float().cpu()
    if waveform.dim() == 3 and waveform.size(0) == 1:
        waveform = waveform.squeeze(0)
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.dim() != 2:
        raise ValueError(f"{role.value} decoded waveform must have shape [channels, time].")
    return waveform


source_waveform = decode_role(Role.SOURCE)
target_waveform = decode_role(Role.TARGET)

print("source waveform shape:", tuple(source_waveform.shape))
display(Audio(source_waveform.numpy(), rate=sample_rate))

print("target waveform shape:", tuple(target_waveform.shape))
display(Audio(target_waveform.numpy(), rate=sample_rate))
